In [5]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
import re

# remplacer par le chemin de ton fichier
df = pd.read_csv("../data/raw/ski-resorts.csv")

# Exemple d'extraction de lat/lon selon format commun "lat, lon" ou "lon lat" ou "POINT(lon lat)"
def parse_coord(s):
    if pd.isna(s):
        return (None, None)
    s = str(s).strip()

    # POINT(lon lat) ou POINT (lon lat)
    m = re.search(r"POINT\s*\(?\s*([-\d\.]+)\s+([-\d\.]+)\s*\)?", s, re.IGNORECASE)
    if m:
        lon = float(m.group(1)); lat = float(m.group(2))
        return lat, lon

    # dict-like / JSON-like: chercher lat et lon par clé (lat, long, lon, lng)
    m_lat = re.search(r"lat['\"]?\s*[:=]\s*['\"]?(-?\d+(?:\.\d+)?)['\"]?", s, re.IGNORECASE)
    m_lon = re.search(r"(?:lon|long|lng)['\"]?\s*[:=]\s*['\"]?(-?\d+(?:\.\d+)?)['\"]?", s, re.IGNORECASE)
    if m_lat and m_lon:
        try:
            return float(m_lat.group(1)), float(m_lon.group(1))
        except:
            pass

    # cas "lat, lon" ou "lon, lat" ou simple liste de deux nombres dans la chaîne
    nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", s)
    if len(nums) >= 2:
        a = float(nums[0]); b = float(nums[1])
        # si la première valeur semble être latitude plausible
        if abs(a) <= 90 and abs(b) <= 180:
            return a, b
        # sinon tenter inversion
        if abs(b) <= 90 and abs(a) <= 180:
            return b, a

    return (None, None)

# remplacer l'assignation et la création de la carte pour utiliser uniquement les lignes avec coords valides
df[['lat','lon']] = df['location_coordinate'].apply(lambda x: pd.Series(parse_coord(x)))

# garder seulement les lignes avec coords valides
df_coords = df.dropna(subset=['lat','lon']).copy()

if df_coords.empty:
    raise ValueError("Aucune coordonnée valide trouvée dans 'location_coordinate'. Vérifiez le format des valeurs (ex: {'lat': '46.1', 'long': '7.2'}).")

# centrer la carte sur les points valides
center = [df_coords['lat'].astype(float).mean(), df_coords['lon'].astype(float).mean()]
m = folium.Map(location=center, zoom_start=4)

# cluster et palette (par pays ou region)
cluster = MarkerCluster().add_to(m)

# palette simple par pays
countries = df_coords['location_country'].fillna("Unknown").unique().tolist()
palette = {c: folium.utilities.color_brewer('Set1', len(countries))[i % 9] if len(countries) <= 9 else None
           for i, c in enumerate(countries)}
# fallback: random colors si palette None
import random
for c in palette:
    if palette[c] is None:
        palette[c] = "#{:06x}".format(random.randint(0, 0xFFFFFF))

for _, row in df_coords.iterrows():
    popup = folium.Popup(f"{row.get('name','')}<br>{row.get('location_country','')} - {row.get('location_region','')}", max_width=300)
    color = palette.get(row.get('location_country','Unknown'))
    folium.CircleMarker(
        location=(float(row['lat']), float(row['lon'])),
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=popup
    ).add_to(cluster)

# sauvegarde
m.save("ski_resorts_map.html")
print("Carte enregistrée sous ski_resorts_map.html")

Carte enregistrée sous ski_resorts_map.html
